In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Inference for GPT

In [ ]:
!pip install -q sentence-transformers
# --- Imports ---
import os
import json
from zipfile import ZipFile
from collections import defaultdict
import pandas as pd
from sentence_transformers import SentenceTransformer
from torch.nn.functional import normalize

# Load lightweight Sentence-BERT model
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-mpnet-base-v2')

# Path to ZIP file
zip_path = "/content/drive/MyDrive/PreferenceTrackerResults/50_persona_updated.zip"

# Set similarity threshold
SIMILARITY_THRESHOLD = 0.4

# Track stats
stats = defaultdict(lambda: {"total": 0, "matched": 0})

# Analyze each JSON in the ZIP
with ZipFile(zip_path, 'r') as archive:
    for file_name in archive.namelist():
        if file_name.endswith(".json"):
            with archive.open(file_name) as f:
                try:
                    data = json.load(f)
                    contradiction = data.get("Contradiction", {})
                    contradiction_type = contradiction.get("type", "unspecified").strip().lower()
                    curr_pref = contradiction.get("curr_pref", "").strip()
                    preferred_response = data.get("Response", {}).get("preferred_response", "").strip()

                    if not curr_pref or not preferred_response:
                        continue

                    stats[contradiction_type]["total"] += 1

                    # Embed and normalize
                    embeddings = model.encode([curr_pref, preferred_response], convert_to_tensor=True)
                    emb_curr = normalize(embeddings[0].unsqueeze(0))
                    emb_resp = normalize(embeddings[1].unsqueeze(0))
                    sim_score = (emb_curr @ emb_resp.T).item()

                    if sim_score >= SIMILARITY_THRESHOLD:
                        stats[contradiction_type]["matched"] += 1

                except Exception as e:
                    print(f"Error processing {file_name}: {e}")

# Summarize results
summary = []
for contradiction_type, counts in stats.items():
    total = counts["total"]
    matched = counts["matched"]
    rate = matched / total if total > 0 else 0.0
    summary.append({
        "Contradiction Type": contradiction_type,
        "Total Examples": total,
        "Matches Curr Pref": matched,
        "Match Rate": round(rate, 3)
    })

# Create and display dataframe
df = pd.DataFrame(summary).sort_values(by="Match Rate", ascending=False)
import IPython
IPython.display.display(df)

Error processing __MACOSX/output/movieRecommendation/._conversation_movieRecommendation_persona24_sample0.json: 'utf-16-be' codec can't decode byte 0xfc in position 162: truncated data
Error processing __MACOSX/output/movieRecommendation/._conversation_movieRecommendation_persona23_sample0.json: 'utf-16-be' codec can't decode byte 0xfc in position 162: truncated data
Error processing __MACOSX/output/movieRecommendation/._conversation_movieRecommendation_persona36_sample0.json: 'utf-16-be' codec can't decode byte 0xfc in position 162: truncated data
Error processing __MACOSX/output/movieRecommendation/._conversation_movieRecommendation_persona31_sample0.json: 'utf-16-be' codec can't decode byte 0xfc in position 162: truncated data
Error processing __MACOSX/output/movieRecommendation/._conversation_movieRecommendation_persona8_sample0.json: 'utf-16-be' codec can't decode byte 0xfc in position 162: truncated data
Error processing __MACOSX/output/movieRecommendation/._conversation_movieRec

,Contradiction Type,Total Examples,Matches Curr Pref,Match Rate
4,topic-specific,11,5,0.455
1,contextual,101,41,0.406
0,temporal,59,22,0.373
2,trade-off,241,79,0.328
5,meta-preference,4,1,0.250
3,ambiguous,22,2,0.091
6,ambiguous_intent,3,0,0.000
7,ambiguous intent,3,0,0.000


In [ ]:


# Load lightweight Sentence-BERT model
model = SentenceTransformer('all-mpnet-base-v2')

# Path to your ZIP file
zip_path = "/content/drive/MyDrive/PreferenceTrackerResults/50_persona_updated.zip"

# Set similarity threshold (can tune later)
SIMILARITY_THRESHOLD = 0.4

# Track stats
stats = defaultdict(lambda: {"total": 0, "matched": 0, "sum_similarity": 0.0})

# Analyze each JSON in the ZIP
with ZipFile(zip_path, 'r') as archive:
    for file_name in archive.namelist():
        if file_name.endswith(".json"):
            with archive.open(file_name) as f:
                try:
                    data = json.load(f)
                    contradiction = data.get("Contradiction", {})
                    contradiction_type = contradiction.get("type", "unspecified").strip().lower()
                    curr_pref = contradiction.get("curr_pref", "").strip()
                    preferred_response = data.get("Response", {}).get("preferred_response", "").strip()

                    if not curr_pref or not preferred_response:
                        continue

                    stats[contradiction_type]["total"] += 1

                    # Embed and normalize
                    embeddings = model.encode([curr_pref, preferred_response], convert_to_tensor=True)
                    emb_curr = normalize(embeddings[0].unsqueeze(0))
                    emb_resp = normalize(embeddings[1].unsqueeze(0))
                    sim_score = (emb_curr @ emb_resp.T).item()

                    # Accumulate similarity sum
                    stats[contradiction_type]["sum_similarity"] += sim_score

                    # Count match
                    if sim_score >= SIMILARITY_THRESHOLD:
                        stats[contradiction_type]["matched"] += 1

                except Exception as e:
                    print(f"Error processing {file_name}: {e}")

# Summarize results
summary = []
for contradiction_type, counts in stats.items():
    total = counts["total"]
    matched = counts["matched"]
    sum_similarity = counts["sum_similarity"]
    rate = matched / total if total > 0 else 0.0
    avg_similarity = sum_similarity / total if total > 0 else 0.0
    summary.append({
        "Contradiction Type": contradiction_type,
        "Total Examples": total,
        "Matches Curr Pref": matched,
        "Match Rate": round(rate, 3),
        "Avg Similarity": round(avg_similarity, 3)
    })

# Create and display dataframe
df = pd.DataFrame(summary).sort_values(by="Match Rate", ascending=False)
import IPython
IPython.display.display(df)


Error processing __MACOSX/output/movieRecommendation/._conversation_movieRecommendation_persona24_sample0.json: 'utf-16-be' codec can't decode byte 0xfc in position 162: truncated data
Error processing __MACOSX/output/movieRecommendation/._conversation_movieRecommendation_persona23_sample0.json: 'utf-16-be' codec can't decode byte 0xfc in position 162: truncated data
Error processing __MACOSX/output/movieRecommendation/._conversation_movieRecommendation_persona36_sample0.json: 'utf-16-be' codec can't decode byte 0xfc in position 162: truncated data
Error processing __MACOSX/output/movieRecommendation/._conversation_movieRecommendation_persona31_sample0.json: 'utf-16-be' codec can't decode byte 0xfc in position 162: truncated data
Error processing __MACOSX/output/movieRecommendation/._conversation_movieRecommendation_persona8_sample0.json: 'utf-16-be' codec can't decode byte 0xfc in position 162: truncated data
Error processing __MACOSX/output/movieRecommendation/._conversation_movieRec

,Contradiction Type,Total Examples,Matches Curr Pref,Match Rate,Avg Similarity
4,topic-specific,11,5,0.455,0.473
1,contextual,101,41,0.406,0.346
0,temporal,59,22,0.373,0.331
2,trade-off,241,79,0.328,0.328
5,meta-preference,4,1,0.250,0.260
3,ambiguous,22,2,0.091,0.246
6,ambiguous_intent,3,0,0.000,0.234
7,ambiguous intent,3,0,0.000,0.167


In [ ]:
# Define output path
output_path = "/content/drive/MyDrive/PreferenceTrackerResults/certainty_analysis_2.csv"

# Save the DataFrame as CSV
df.to_csv(output_path, index=False)

print(f"✅ Results saved to: {output_path}")

✅ Results saved to: /content/drive/MyDrive/PreferenceTrackerResults/certainty_analysis_2.csv


Inference for Qwen

In [ ]:
!pip install -q sentence-transformers

# --- Imports ---
import os
import json
from zipfile import ZipFile
from collections import defaultdict
import pandas as pd
from sentence_transformers import SentenceTransformer
from torch.nn.functional import normalize



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 81.1 MB/s eta 0:00:00


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: '/content/qwen_outputs.zip'

In [ ]:
# Load lightweight Sentence-BERT model
model = SentenceTransformer('all-mpnet-base-v2')

# Path to your ZIP file — your new generated outputs!
zip_path = "/content/drive/MyDrive/qwen_outputs_reasoning.zip"

# Set similarity threshold (if you still want match rate)
SIMILARITY_THRESHOLD = 0.4

# Track stats
stats = defaultdict(lambda: {"total": 0, "matched": 0, "sum_similarity": 0.0})

# Analyze each JSON in the ZIP
with ZipFile(zip_path, 'r') as archive:
    for file_name in archive.namelist():
        if file_name.endswith(".json"):
            with archive.open(file_name) as f:
                try:
                    data = json.load(f)

                    agent_answer = data.get("AgentAnswer", "").strip()
                    preferred_response = data.get("preferred_response", "").strip()
                    curr_pref = data.get("curr_pref", "").strip()

                    contradiction_type = "overall"  # since your output does not store type anymore

                    if not agent_answer or not preferred_response:
                        continue

                    stats[contradiction_type]["total"] += 1

                    # Embed and normalize
                    embeddings = model.encode([agent_answer, preferred_response], convert_to_tensor=True)
                    emb_agent = normalize(embeddings[0].unsqueeze(0))
                    emb_pref = normalize(embeddings[1].unsqueeze(0))
                    sim_score = (emb_agent @ emb_pref.T).item()

                    # Accumulate similarity sum
                    stats[contradiction_type]["sum_similarity"] += sim_score

                    # Also still count match if desired
                    if sim_score >= SIMILARITY_THRESHOLD:
                        stats[contradiction_type]["matched"] += 1

                except Exception as e:
                    print(f"Error processing {file_name}: {e}")

# Summarize results
summary = []
for contradiction_type, counts in stats.items():
    total = counts["total"]
    matched = counts["matched"]
    sum_similarity = counts["sum_similarity"]
    rate = matched / total if total > 0 else 0.0
    avg_similarity = sum_similarity / total if total > 0 else 0.0
    summary.append({
        "Contradiction Type": contradiction_type,
        "Total Examples": total,
        "Matches Preferred Response": matched,
        "Match Rate": round(rate, 3),
        "Avg Similarity": round(avg_similarity, 3)
    })



In [ ]:
# Create and display dataframe
df = pd.DataFrame(summary).sort_values(by="Avg Similarity", ascending=False)
import IPython
IPython.display.display(df)

# Save dataframe as CSV
df_path = "/content/qwen_eval_results.csv"
df.to_csv(df_path, index=False)

# Copy to Google Drive
!cp /content/qwen_eval_results.csv /content/drive/MyDrive/qwen_eval_results.csv

print("Saved evaluation results to Google Drive → qwen_eval_results.csv")


,Contradiction Type,Total Examples,Matches Preferred Response,Match Rate,Avg Similarity
0,overall,88,38,0.432,0.362


Saved evaluation results to Google Drive → qwen_eval_results.csv
